In [17]:
import pandas as pd
from collections import Counter
import numpy as np
import re

In [18]:
df_x_ray = pd.read_csv('mimic_cxr_translated.csv')
df_x_ray.head()

,original_report,patient_friendly_translation,detected_diseases
0,Findings: Impression: Compared to chest radio...,This chest X-ray shows a few findings: there i...,"Pleural Effusion, Pulmonary Edema / Vascular C..."
1,Findings: Impression:,"Unfortunately, no detailed findings were recor...",No acute cardiopulmonary disease
2,Findings: No previous images. There are relat...,This chest X-ray shows a few findings: there i...,"Pleural Effusion, Pneumonia"
3,Findings: As compared to the previous radiogra...,This chest X-ray shows some findings that don'...,No acute cardiopulmonary disease
4,Findings: Mild pulmonary vascular congestion w...,This chest X-ray shows a few findings: the hea...,"Cardiomegaly, Pulmonary Edema / Vascular Conge..."


In [19]:
df_ct = pd.read_csv('translated_ct_reports.csv')
df_ct.head()

,report,translation,diseases
0,1.5 mm thick non-contrast sections were taken ...,This is a completely normal chest scan. The ai...,Normal / No acute disease
1,1.5 mm thick non-contrast sections were taken ...,The heart is slightly enlarged and the main bl...,"Cardiomegaly, Mild aortic dilation, Hiatal her..."
2,Non-contrast images were taken in the axial pl...,This scan shows hardening of the arteries arou...,"Atherosclerosis (coronary and aortic), Emphyse..."
3,Non-contrast images were taken in the axial pl...,This scan shows hardened plaques in the heart'...,"Atherosclerosis, Hiatal hernia, Pulmonary fibr..."
4,"Before IVKM was given, sections were taken in ...",This scan of the airways shows widened and thi...,"Bronchiectasis, Possible infective pulmonary p..."


In [20]:
# rename to common schema
df_x_ray = df_x_ray.rename(columns={
    'original_report': 'report',
    'patient_friendly_translation': 'translation',
    'detected_diseases': 'diseases'
})[['report', 'translation', 'diseases']]

df_ct = df_ct.rename(columns={
    'report': 'report',
    'translation': 'translation',
    'diseases': 'diseases'
})[['report', 'translation', 'diseases']]

df = pd.concat([df_x_ray, df_ct], ignore_index=True)
df = df.dropna(subset=['report', 'translation', 'diseases']).reset_index(drop=True)
print(df.shape)
df.head()

(1050, 3)


,report,translation,diseases
0,Findings: Impression: Compared to chest radio...,This chest X-ray shows a few findings: there i...,"Pleural Effusion, Pulmonary Edema / Vascular C..."
1,Findings: Impression:,"Unfortunately, no detailed findings were recor...",No acute cardiopulmonary disease
2,Findings: No previous images. There are relat...,This chest X-ray shows a few findings: there i...,"Pleural Effusion, Pneumonia"
3,Findings: As compared to the previous radiogra...,This chest X-ray shows some findings that don'...,No acute cardiopulmonary disease
4,Findings: Mild pulmonary vascular congestion w...,This chest X-ray shows a few findings: the hea...,"Cardiomegaly, Pulmonary Edema / Vascular Conge..."


In [21]:
# split on comma, strip whitespace
df['disease_list'] = df['diseases'].apply(lambda x: [d.strip() for d in x.split(',')])

all_diseases = Counter()
for dl in df['disease_list']:
    all_diseases.update(dl)

all_diseases.most_common(30)

[('No acute cardiopulmonary disease', 153),
 ('Pneumothorax', 122),
 ('nonspecific)', 122),
 ('Atelectasis', 115),
 ('Pulmonary nodules (small', 108),
 ('Pleural Effusion', 104),
 ('Cardiomegaly', 93),
 ('Pulmonary Edema / Vascular Congestion', 84),
 ('Pneumonia', 71),
 ('Coronary and aortic atherosclerosis', 69),
 ('Normal / No acute disease', 53),
 ('Consolidation', 40),
 ('Spinal Degenerative Changes', 33),
 ('Pulmonary fibrosis/scarring (sequelae)', 32),
 ('Pulmonary nodule (small', 31),
 ('Hiatal hernia (sliding)', 31),
 ('Covid-19 pneumonia (consistent findings)', 30),
 ('Emphysema', 29),
 ('Mild emphysema', 29),
 ('recommend clinical correlation)', 28),
 ('Hepatic steatosis (fatty liver)', 27),
 ('Coronary atherosclerosis', 26),
 ('Emphysema/COPD', 25),
 ('Pericardial effusion (minimal)', 24),
 ('Mild hepatic steatosis', 23),
 ('nonspecific', 22),
 ('Mosaic attenuation pattern (possible small airway/vessel disease)', 20),
 ('Pulmonary fibrosis/scarring (apical sequelae)', 20),
 

In [22]:
def smart_split(diseases_str):
    """Split on commas, but not inside parentheses."""
    # protect commas inside parentheses by temporarily replacing them
    parts = []
    depth = 0
    current = []
    for ch in diseases_str:
        if ch == '(':
            depth += 1
            current.append(ch)
        elif ch == ')':
            depth -= 1
            current.append(ch)
        elif ch == ',' and depth == 0:
            parts.append(''.join(current).strip())
            current = []
        else:
            current.append(ch)
    if current:
        parts.append(''.join(current).strip())
    return [p for p in parts if p]

def clean_label(label):
    # remove markdown link artifacts like [/](https://...)
    label = re.sub(r'\[.*?\]\(.*?\)', '/', label)
    # collapse multiple spaces
    label = re.sub(r'\s+', ' ', label).strip()
    return label

In [23]:
NORMALIZATION_MAP = {
    'No acute cardiopulmonary disease': 'No acute cardiopulmonary disease',
    'Normal / No acute disease': 'No acute cardiopulmonary disease',

    'Pneumothorax': 'Pneumothorax',

    'Atelectasis': 'Atelectasis',

    'Pulmonary nodules (small': 'Pulmonary nodules',
    'Pulmonary nodule (small': 'Pulmonary nodules',
    'nonspecific)': None,   # drop — fragment, not a disease
    'nonspecific': None,    # drop
    'recommend clinical correlation)': None,  # drop

    'Pleural Effusion': 'Pleural Effusion',

    'Cardiomegaly': 'Cardiomegaly',

    'Pulmonary Edema / Vascular Congestion': 'Pulmonary Edema/Vascular Congestion',
    'Pulmonary Edema': 'Pulmonary Edema/Vascular Congestion',
    'Vascular Congestion': 'Pulmonary Edema/Vascular Congestion',

    'Pneumonia': 'Pneumonia',
    'Covid-19 pneumonia (consistent findings)': 'Pneumonia',

    'Coronary and aortic atherosclerosis': 'Atherosclerosis',
    'Coronary atherosclerosis': 'Atherosclerosis',

    'Consolidation': 'Consolidation',

    'Spinal Degenerative Changes': 'Spinal Degenerative Changes',

    'Pulmonary fibrosis/scarring (sequelae)': 'Pulmonary Fibrosis/Scarring',
    'Pulmonary fibrosis/scarring (apical sequelae)': 'Pulmonary Fibrosis/Scarring',

    'Hiatal hernia (sliding)': 'Hiatal Hernia',
    'Hiatal hernia': 'Hiatal Hernia',

    'Emphysema': 'Emphysema/COPD',
    'Mild emphysema': 'Emphysema/COPD',
    'Emphysema/COPD': 'Emphysema/COPD',

    'Hepatic steatosis (fatty liver)': 'Hepatic Steatosis',
    'Mild hepatic steatosis': 'Hepatic Steatosis',

    'Pericardial effusion (minimal)': 'Pericardial Effusion',

    'Mosaic attenuation pattern (possible small airway/vessel disease)': 'Mosaic Attenuation Pattern',

    'Rib/Bone Fracture': 'Rib/Bone Fracture',
}

def normalize_label(label):
    label = clean_label(label)
    if label in NORMALIZATION_MAP:
        return NORMALIZATION_MAP[label]
    # fallback: return as-is, but flag for review
    return label

In [24]:
def process_diseases(diseases_str):
    raw_parts = smart_split(diseases_str)
    cleaned = []
    for p in raw_parts:
        norm = normalize_label(p)
        if norm is not None:
            cleaned.append(norm)
    return cleaned

df['disease_list'] = df['diseases'].apply(process_diseases)

all_diseases = Counter()
for dl in df['disease_list']:
    all_diseases.update(dl)

sorted(all_diseases.items(), key=lambda x: -x[1])

[('No acute cardiopulmonary disease', 206),
 ('Pneumothorax', 122),
 ('Atelectasis', 115),
 ('Pleural Effusion', 104),
 ('Pneumonia', 101),
 ('Atherosclerosis', 96),
 ('Cardiomegaly', 93),
 ('Pulmonary Edema/Vascular Congestion', 84),
 ('Emphysema/COPD', 83),
 ('Pulmonary nodules (small, nonspecific)', 74),
 ('Pulmonary Fibrosis/Scarring', 52),
 ('Hiatal Hernia', 50),
 ('Hepatic Steatosis', 50),
 ('Consolidation', 40),
 ('Spinal Degenerative Changes', 33),
 ('Pericardial Effusion', 24),
 ('Mosaic Attenuation Pattern', 20),
 ('Rib/Bone Fracture', 19),
 ('Pulmonary Nodule', 16),
 ('Pulmonary artery enlargement', 14),
 ('Pulmonary nodule (small, nonspecific)', 14),
 ('Cholelithiasis', 14),
 ('Hiatal hernia (mild)', 14),
 ('Possible Aspiration', 13),
 ('Aortic atherosclerosis', 13),
 ('Bilateral pleural effusion', 12),
 ('Possible thymic remnant tissue (benign)', 12),
 ('Status post cholecystectomy', 12),
 ('Mediastinal/hilar lymphadenopathy', 11),
 ('Possible Malignancy/Mass', 10),
 ('Acc

In [25]:
NORMALIZATION_MAP.update({
    'Pulmonary nodules (small, nonspecific)': 'Pulmonary Nodules',
    'Pulmonary nodule (small, nonspecific)': 'Pulmonary Nodules',
    'Pulmonary Nodule': 'Pulmonary Nodules',

    'Hiatal hernia (mild)': 'Hiatal Hernia',

    'Aortic atherosclerosis': 'Atherosclerosis',

    'Pulmonary artery enlargement': 'Pulmonary Artery Enlargement',
    'Possible Aspiration': 'Possible Aspiration',
    'Cholelithiasis': 'Cholelithiasis',
})

In [26]:
MIN_COUNT = 5
unmapped_significant = {d: c for d, c in all_diseases.items() 
                        if c >= MIN_COUNT and d not in NORMALIZATION_MAP.values() 
                        and d not in NORMALIZATION_MAP}
unmapped_significant

{'Hiatal/Diaphragmatic Hernia': 5,
 'Possible Malignancy/Mass': 10,
 'Pulmonary fibrosis/scarring': 6,
 'Degenerative bone changes': 7,
 'Degenerative spinal changes': 8,
 'Reduced bone density': 5,
 'Mediastinal lymphadenopathy': 6,
 'Bilateral pleural effusion': 12,
 'Pulmonary nodules (stable)': 5,
 'Aortic dilation (ascending aorta)': 6,
 'Splenomegaly': 6,
 'Aortic and coronary atherosclerosis': 7,
 'Bilateral pleural effusion (minimal)': 7,
 'Thoracic spondylosis': 5,
 'Pericardial effusion': 9,
 'Osteoporosis': 6,
 'Possible thymic remnant tissue (benign)': 12,
 'Mild bronchiectasis': 5,
 'Accessory spleen (normal variant)': 10,
 'Mild bronchial wall thickening': 6,
 'Mosaic attenuation pattern': 9,
 'Mild cardiomegaly': 5,
 'Mediastinal/hilar lymphadenopathy': 11,
 'Mild aortic dilation (ascending aorta)': 6,
 'Status post cholecystectomy': 12,
 'Mild scoliosis': 8,
 'Mild atelectasis': 7,
 'Bilateral pleural effusion with atelectasis': 6,
 'Mild coronary atherosclerosis': 7,
 

In [27]:
NORMALIZATION_MAP.update({
    # Hernia
    'Hiatal/Diaphragmatic Hernia': 'Hiatal Hernia',
    'Hiatal hernia (mild, sliding)': 'Hiatal Hernia',

    # Fibrosis
    'Pulmonary fibrosis/scarring': 'Pulmonary Fibrosis/Scarring',

    # Spine/bone
    'Degenerative bone changes': 'Spinal Degenerative Changes',
    'Degenerative spinal changes': 'Spinal Degenerative Changes',
    'Thoracic spondylosis': 'Spinal Degenerative Changes',
    'Mild thoracic spondylosis': 'Spinal Degenerative Changes',
    'Reduced bone density': 'Osteoporosis',
    'Osteoporosis': 'Osteoporosis',

    # Lymphadenopathy
    'Mediastinal lymphadenopathy': 'Lymphadenopathy',
    'Mediastinal/hilar lymphadenopathy': 'Lymphadenopathy',

    # Pleural effusion
    'Bilateral pleural effusion': 'Pleural Effusion',
    'Bilateral pleural effusion (minimal)': 'Pleural Effusion',

    # Nodules
    'Pulmonary nodules (stable)': 'Pulmonary Nodules',

    # Aorta
    'Aortic dilation (ascending aorta)': 'Aortic Dilation',
    'Mild aortic dilation (ascending aorta)': 'Aortic Dilation',

    # Atherosclerosis
    'Aortic and coronary atherosclerosis': 'Atherosclerosis',
    'Mild coronary atherosclerosis': 'Atherosclerosis',

    # Pericardial effusion
    'Pericardial effusion': 'Pericardial Effusion',

    # Cardiomegaly
    'Mild cardiomegaly': 'Cardiomegaly',

    # Mosaic attenuation
    'Mosaic attenuation pattern': 'Mosaic Attenuation Pattern',

    # Bronchiectasis / bronchial
    'Mild bronchiectasis': 'Bronchiectasis',
    'Mild bronchial wall thickening': 'Bronchial Wall Thickening',

    # Pneumonia
    'Covid-19 pneumonia (typical/probable findings)': 'Pneumonia',

    # Splenomegaly / spleen
    'Splenomegaly': 'Splenomegaly',
    'Accessory spleen (normal variant)': None,  # incidental normal variant, drop

    # Misc — drop as non-actionable/incidental/too rare to generalize
    'Possible Malignancy/Mass': 'Possible Malignancy/Mass',
    'Possible thymic remnant tissue (benign)': None,  # incidental benign finding, drop
    'Status post cholecystectomy': None,  # surgical history, not a disease finding
    'Right nephrolithiasis (small)': 'Nephrolithiasis',
})

## 📦 Report Generalization Pipeline Setup & Model Training

In [28]:
import torch
import numpy as np
import pandas as pd
from collections import Counter
from sklearn.model_selection import train_test_split
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification, 
    AutoModelForSeq2SeqLM, 
    Trainer, 
    TrainingArguments, 
    Seq2SeqTrainer, 
    Seq2SeqTrainingArguments, 
    DataCollatorForSeq2Seq
)
from torch.utils.data import Dataset
from peft import LoraConfig, get_peft_model, TaskType
import evaluate
from sklearn.metrics import f1_score

# Device configuration
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")

Using device: mps


### 1. Data Cleaning & Multi-Hot Encoding

In [29]:
# Process disease labels using process_diseases function defined earlier
df['disease_list'] = df['diseases'].apply(process_diseases)

# Filter low frequency diseases
MIN_COUNT = 8
all_diseases = Counter()
for dl in df['disease_list']:
    all_diseases.update(dl)

label_vocab = sorted([d for d, c in all_diseases.items() if c >= MIN_COUNT])
label2idx = {d: i for i, d in enumerate(label_vocab)}
num_labels = len(label_vocab)

print(f"Total canonical diseases with count >= {MIN_COUNT}: {num_labels}")
print("Label Vocab:", label_vocab)

# Multi-hot encode the disease list
def multi_hot_encode(disease_list):
    vec = np.zeros(num_labels, dtype=np.float32)
    for d in disease_list:
        if d in label2idx:
            vec[label2idx[d]] = 1.0
    return vec

df['label_vec'] = df['disease_list'].apply(multi_hot_encode)
df.head()

Total canonical diseases with count >= 8: 26
Label Vocab: ['Aortic Dilation', 'Atelectasis', 'Atherosclerosis', 'Cardiomegaly', 'Cholelithiasis', 'Consolidation', 'Emphysema/COPD', 'Hepatic Steatosis', 'Hiatal Hernia', 'Lymphadenopathy', 'Mild scoliosis', 'Mosaic Attenuation Pattern', 'No acute cardiopulmonary disease', 'Osteoporosis', 'Pericardial Effusion', 'Pleural Effusion', 'Pneumonia', 'Pneumothorax', 'Possible Aspiration', 'Possible Malignancy/Mass', 'Pulmonary Artery Enlargement', 'Pulmonary Edema/Vascular Congestion', 'Pulmonary Fibrosis/Scarring', 'Pulmonary Nodules', 'Rib/Bone Fracture', 'Spinal Degenerative Changes']


,report,translation,diseases,disease_list,label_vec
0,Findings: Impression: Compared to chest radio...,This chest X-ray shows a few findings: there i...,"Pleural Effusion, Pulmonary Edema / Vascular C...","[Pleural Effusion, Pulmonary Edema/Vascular Co...","[0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
1,Findings: Impression:,"Unfortunately, no detailed findings were recor...",No acute cardiopulmonary disease,[No acute cardiopulmonary disease],"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
2,Findings: No previous images. There are relat...,This chest X-ray shows a few findings: there i...,"Pleural Effusion, Pneumonia","[Pleural Effusion, Pneumonia]","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
3,Findings: As compared to the previous radiogra...,This chest X-ray shows some findings that don'...,No acute cardiopulmonary disease,[No acute cardiopulmonary disease],"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
4,Findings: Mild pulmonary vascular congestion w...,This chest X-ray shows a few findings: the hea...,"Cardiomegaly, Pulmonary Edema / Vascular Conge...","[Cardiomegaly, Pulmonary Edema/Vascular Conges...","[0.0, 0.0, 0.0, 1.0, 0.0, 1.0, 0.0, 0.0, 0.0, ..."


### 2. Train / Val / Test Split

In [30]:
# Train-val-test split (80/10/10)
train_df, val_test_df = train_test_split(df, test_size=0.20, random_state=42)
val_df, test_df = train_test_split(val_test_df, test_size=0.50, random_state=42)

print(f"Train samples: {len(train_df)}")
print(f"Validation samples: {len(val_df)}")
print(f"Test samples: {len(test_df)}")

Train samples: 840
Validation samples: 105
Test samples: 105


### 3. Model A — Disease Classifier (ClinicalBERT)

In [31]:
class DiseaseClassifierDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=512):
        self.reports = df['report'].tolist()
        self.labels = df['label_vec'].tolist()
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.reports)

    def __getitem__(self, idx):
        report = str(self.reports[idx])
        label = self.labels[idx]
        
        inputs = self.tokenizer(
            report,
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        
        item = {key: val.squeeze(0) for key, val in inputs.items()}
        item['labels'] = torch.tensor(label, dtype=torch.float32)
        return item

In [ ]:
# Model A: Bio_ClinicalBERT
CLINICALBERT_MODEL = "emilyalsentzer/Bio_ClinicalBERT"
tokenizer_a = AutoTokenizer.from_pretrained(CLINICALBERT_MODEL)
model_a = AutoModelForSequenceClassification.from_pretrained(
    CLINICALBERT_MODEL, 
    num_labels=num_labels, 
    problem_type="multi_label_classification"
).to(device)

train_dataset_a = DiseaseClassifierDataset(train_df, tokenizer_a)
val_dataset_a = DiseaseClassifierDataset(val_df, tokenizer_a)

def compute_metrics_classifier(eval_pred):
    logits, labels = eval_pred
    probs = 1 / (1 + np.exp(-logits))  # Sigmoid
    preds = (probs > 0.5).astype(float)
    
    f1_micro = f1_score(labels, preds, average='micro', zero_division=0)
    f1_macro = f1_score(labels, preds, average='macro', zero_division=0)
    
    return {
        "f1_micro": f1_micro,
        "f1_macro": f1_macro
    }

training_args_a = TrainingArguments(
    output_dir="./clinicalbert_disease_classifier",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=5,
    learning_rate=2e-5,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,
    report_to="none",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
)

trainer_a = Trainer(
    model=model_a,
    args=training_args_a,
    train_dataset=train_dataset_a,
    eval_dataset=val_dataset_a,
    compute_metrics=compute_metrics_classifier,
)

trainer_a.train()
model_a.save_pretrained("./disease_classifier_model", safe_serialization=False)
tokenizer_a.save_pretrained("./disease_classifier_model")

# Save in PyTorch dict format like the other VLM model
torch.save({
    "model_state_dict": model_a.state_dict(),
    "label_vocab": label_vocab,
    "label2idx": label2idx
}, "./disease_classifier_model.pt")
print("Saved Model A in Hugging Face format (with pytorch_model.bin) and PyTorch dict format (disease_classifier_model.pt).")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: emilyalsentzer/Bio_ClinicalBERT
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the chec

Epoch,Training Loss,Validation Loss,F1 Micro,F1 Macro
1,0.278141,0.231923,0.000000,0.000000
2,0.204675,0.201987,0.000000,0.000000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/Users/md.nurealamsiddiquee/Projects/lumora/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/Users/md.nurealamsiddiquee/Projects/lumora/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


### 4. Model B — Translation Model (T5 + LoRA)

In [ ]:
class TranslationDataset(Dataset):
    def __init__(self, df, tokenizer, max_input_len=512, max_target_len=256):
        self.reports = df['report'].tolist()
        self.translations = df['translation'].tolist()
        self.tokenizer = tokenizer
        self.max_input_len = max_input_len
        self.max_target_len = max_target_len

    def __len__(self):
        return len(self.reports)

    def __getitem__(self, idx):
        input_text = "translate to patient-friendly: " + str(self.reports[idx])
        target_text = str(self.translations[idx])

        model_inputs = self.tokenizer(
            input_text, truncation=True, max_length=self.max_input_len
        )
        labels = self.tokenizer(
            target_text, truncation=True, max_length=self.max_target_len
        )
        model_inputs["labels"] = labels["input_ids"]
        return model_inputs

In [ ]:
T5_MODEL = "t5-small"
t5_tokenizer = AutoTokenizer.from_pretrained(T5_MODEL)
t5_model = AutoModelForSeq2SeqLM.from_pretrained(T5_MODEL)

# LoRA for low VRAM
lora_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=["q", "v"]  # T5 attention projections
)
t5_model = get_peft_model(t5_model, lora_config)
t5_model.print_trainable_parameters()

train_t5_ds = TranslationDataset(train_df, t5_tokenizer)
val_t5_ds = TranslationDataset(val_df, t5_tokenizer)

data_collator = DataCollatorForSeq2Seq(t5_tokenizer, model=t5_model)

training_args_b = Seq2SeqTrainingArguments(
    output_dir="./t5_translation",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    num_train_epochs=5,
    learning_rate=3e-4,
    fp16=torch.cuda.is_available(),
    eval_strategy="epoch",
    save_strategy="epoch",
    predict_with_generate=True,
    logging_steps=50,
    gradient_checkpointing=True,
    report_to="none",
)

trainer_b = Seq2SeqTrainer(
    model=t5_model,
    args=training_args_b,
    train_dataset=train_t5_ds,
    eval_dataset=val_t5_ds,
    data_collator=data_collator,
    processing_class=t5_tokenizer,
)

trainer_b.train()
t5_model.save_pretrained("./translation_model", safe_serialization=False)
t5_tokenizer.save_pretrained("./translation_model")

# Save in PyTorch dict format like the other VLM model
torch.save({
    "model_state_dict": t5_model.state_dict(),
}, "./translation_model.pt")
print("Saved Model B in Hugging Face format (with adapter_model.bin) and PyTorch dict format (translation_model.pt).")

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

trainable params: 589,824 || all params: 61,096,448 || trainable%: 0.9654


TypeError: Seq2SeqTrainer.__init__() got an unexpected keyword argument 'tokenizer'

### 5. Comprehensive Model Evaluation (Model A & Model B)

In [ ]:
# =====================================================================
# 5.1 Load Saved PyTorch Format Models (Sanity & Verification)
# =====================================================================
print("🔄 Verifying PyTorch format models can be loaded correctly...")

# Load Model A (Disease Classifier) from PyTorch format (pytorch_model.bin)
loaded_model_a = AutoModelForSequenceClassification.from_pretrained(
    "./disease_classifier_model",
    num_labels=num_labels,
    problem_type="multi_label_classification"
).to(device)
print("✅ Model A loaded successfully from PyTorch format (pytorch_model.bin).")

# Load Model B (Translation Model with LoRA Adapter) from PyTorch format (adapter_model.bin)
from peft import PeftModel
base_t5 = AutoModelForSeq2SeqLM.from_pretrained(T5_MODEL)
loaded_t5_model = PeftModel.from_pretrained(base_t5, "./translation_model").to(device)
print("✅ Model B loaded successfully from PyTorch format (adapter_model.bin).")

# =====================================================================
# 5.2 Evaluate Model A (Disease Classifier) on Test Set
# =====================================================================
print("\n📊 Evaluating Model A (Disease Classifier) on test set...")
from torch.utils.data import DataLoader
from sklearn.metrics import precision_score, recall_score

test_dataset_a = DiseaseClassifierDataset(test_df, tokenizer_a)
test_loader_a = DataLoader(test_dataset_a, batch_size=8, shuffle=False)

loaded_model_a.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for batch in test_loader_a:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels']
        
        logits = loaded_model_a(input_ids=input_ids, attention_mask=attention_mask).logits
        probs = torch.sigmoid(logits).cpu().numpy()
        preds = (probs > 0.5).astype(float)
        
        all_preds.append(preds)
        all_labels.append(labels.numpy())

all_preds = np.vstack(all_preds)
all_labels = np.vstack(all_labels)

# Compute metrics
f1_micro_test = f1_score(all_labels, all_preds, average='micro', zero_division=0)
f1_macro_test = f1_score(all_labels, all_preds, average='macro', zero_division=0)
precision_test = precision_score(all_labels, all_preds, average='micro', zero_division=0)
recall_test = recall_score(all_labels, all_preds, average='micro', zero_division=0)

print("Disease Classifier (Model A) Test Metrics:")
print(f"  F1 (Micro): {f1_micro_test:.4f}")
print(f"  F1 (Macro): {f1_macro_test:.4f}")
print(f"  Precision (Micro): {precision_test:.4f}")
print(f"  Recall (Micro): {recall_test:.4f}")

# =====================================================================
# 5.3 Evaluate Model B (Translation Model) on Test Set
# =====================================================================
print("\n📊 Evaluating Model B (Translation Model) on test subset...")
rouge_metric = evaluate.load("rouge")

# Evaluate on a subset of the test set for speed (e.g., 50 samples)
eval_subset = test_df.head(50)
predictions = []
references = eval_subset['translation'].tolist()

loaded_t5_model.eval()
for report in eval_subset['report']:
    input_text = "translate to patient-friendly: " + str(report)
    inputs = t5_tokenizer(input_text, return_tensors="pt", truncation=True, max_length=512).to(device)
    with torch.no_grad():
        outputs = loaded_t5_model.generate(**inputs, max_length=256, num_beams=4)
    pred_text = t5_tokenizer.decode(outputs[0], skip_special_tokens=True)
    predictions.append(pred_text)

rouge_results = rouge_metric.compute(predictions=predictions, references=references)
print("Translation Model (Model B) Test ROUGE Metrics:")
for k, v in rouge_results.items():
    print(f"  {k}: {v:.4f}")

### 6. Combined Inference Pipeline

In [ ]:
def predict(report_text, disease_model, disease_tokenizer, translation_model, translation_tokenizer, threshold=0.5):
    # disease prediction
    disease_model.eval()
    enc = disease_tokenizer(report_text, truncation=True, max_length=512, return_tensors="pt").to(device)
    with torch.no_grad():
        logits = disease_model(**enc).logits
        probs = torch.sigmoid(logits).squeeze(0).cpu().numpy()
    predicted_diseases = [label_vocab[i] for i, p in enumerate(probs) if p > threshold]

    # translation
    translation_model.eval()
    input_text = "translate to patient-friendly: " + report_text
    t5_enc = translation_tokenizer(input_text, truncation=True, max_length=512, return_tensors="pt").to(device)
    with torch.no_grad():
        gen_ids = translation_model.generate(**t5_enc, max_length=256, num_beams=4)
    translation = translation_tokenizer.decode(gen_ids[0], skip_special_tokens=True)

    return {"diseases": predicted_diseases, "translation": translation}

# Test combined inference function on a sample report from the test set
sample_row = test_df.iloc[0]
sample_report = sample_row['report']
print("--- Sample Input Report ---")
print(sample_report)
print("\n--- Ground Truth Translation ---")
print(sample_row['translation'])
print("\n--- Predicted Output ---")
res = predict(sample_report, model_a, tokenizer_a, t5_model, t5_tokenizer)
print("Predicted Diseases:", res["diseases"])
print("Translation:", res["translation"])